In [0]:
import logging
from pyspark.sql import functions as F

In [0]:
%pip install openpyxl

In [0]:

# Configuração de logs
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("RawVeiculosEletricos")

base_path = "/Volumes/workspace/default/raw_veiculos_eletricos"

arquivos_csv = {
    "raw_emplacamentos_veiculos": f"{base_path}/recarregados-emplacamentos-eletricos-hibridos.csv",
    "raw_ipva_veiculos_2026": f"{base_path}/recarregados-ipva-carros-eletricos-hibridos-2026.csv",
    "raw_ficha_tecnica_veiculos": f"{base_path}/recarregados-carros-eletricos-hibridos-ficha.csv"
}

for nome_tabela, caminho_arquivo in arquivos_csv.items():
    logger.info(f"Lendo arquivo CSV: {caminho_arquivo}")
    
    # Lê o CSV mantendo tudo como string na Raw para evitar falha de parse e preserva cabeçalho
    df_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(caminho_arquivo)
        .withColumn("_data_ingestao", F.current_timestamp())
    )
    
    # Limpa eventuais espaços ou caracteres inválidos nos nomes das colunas para o Delta aceitar
    colunas_limpas = [
        c.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_")
        for c in df_raw.columns
    ]
    df_raw = df_raw.toDF(*colunas_limpas)
    
    logger.info(f"Gravando tabela Delta '{nome_tabela}' ({df_raw.count()} registros)...")
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

logger.info("Ingestão dos 3 arquivos CSV concluída com sucesso!")

In [0]:
import pandas as pd

caminho_excel = f"{base_path}/D_Frota_por_UF_Municipio_COMBUSTIVEL_Outubro_2026.xlsx"
logger.info(f"Lendo planilha completa da SENATRAN: {caminho_excel}")

# Lê tudo como string (dtype=str) para preservar os dados brutos na camada Raw sem erro de tipagem
pdf_senatran = pd.read_excel(caminho_excel, header=0, dtype=str)

# Converte de Pandas para Spark DataFrame
df_raw_senatran = (
    spark.createDataFrame(pdf_senatran)
    .toDF("uf", "municipio", "combustivel", "qtd_veiculos")
    .withColumn("_data_ingestao", F.current_timestamp())
)

tabela_senatran = "raw_frota_senatran"
logger.info(f"Gravando tabela Delta '{tabela_senatran}' ({df_raw_senatran.count()} registros)...")

(
    df_raw_senatran.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_senatran)
)

logger.info("Camada RAW 100% concluída! Todas as 4 tabelas brutas foram ingeridas.")